# Random Graphs — Exercises

> *"The giant component appears as suddenly as a phase transition, when average degree crosses one."*

8 graded exercises from ★ (accessible) to ★★★ (research-level).

**Companion:** [notes.md](notes.md) | [theory.ipynb](theory.ipynb)

In [ ]:
import numpy as np
from scipy.optimize import brentq

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    try:
        import seaborn as sns
        sns.set_theme(style='whitegrid', palette='colorblind')
    except ImportError:
        plt.style.use('seaborn-v0_8-whitegrid')
    mpl.rcParams.update({
        'figure.figsize': (10, 6), 'figure.dpi': 120,
        'font.size': 13, 'axes.titlesize': 15, 'axes.labelsize': 13,
        'xtick.labelsize': 11, 'ytick.labelsize': 11,
        'legend.fontsize': 11, 'lines.linewidth': 2.0,
        'axes.spines.top': False, 'axes.spines.right': False,
    })
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

COLORS = {
    'primary': '#0077BB', 'secondary': '#EE7733',
    'tertiary': '#009988', 'error': '#CC3311',
    'neutral': '#555555', 'highlight': '#EE3377',
}

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title, stars):
    print(f'\n{"="*60}')
    print(f'{title}  [{stars}]')
    print('='*60)

def check_close(computed, expected, msg, atol=1e-3):
    ok = abs(computed - expected) <= atol + 1e-8
    status = 'PASS' if ok else 'FAIL'
    print(f'{status}: {msg} | got {computed:.6f}, expected ~{expected:.6f}')
    return ok

def check_true(condition, msg):
    status = 'PASS' if condition else 'FAIL'
    print(f'{status}: {msg}')
    return condition

print('Setup complete.')

---

## Exercise 1 ★ — Phase Transition Simulation

Simulate the Erdős-Rényi phase transition for $n = 1000$ nodes.

**(a)** Implement a $G(n,p)$ sampler.

**(b)** For $c \in \{0.5, 0.8, 1.0, 1.5, 2.0, 3.0\}$ with $p = c/n$, compute $L_1/n$ (fraction of nodes in the largest component).

**(c)** Compute the theoretical giant component fraction $\beta(c)$ satisfying $\beta = 1 - e^{-c\beta}$ (solve numerically).

**(d)** Plot $L_1/n$ vs $c$ with the theoretical curve overlaid.

In [ ]:
# Exercise 1 — YOUR CODE HERE

def gnp_sample(n, p, seed=None):
    """Sample G(n,p) adjacency matrix (symmetric, zero diagonal)."""
    # YOUR CODE HERE
    pass

def bfs_largest_component(A):
    """Return size of largest connected component via BFS."""
    # YOUR CODE HERE
    pass

def beta_theory(c):
    """Solve beta = 1 - exp(-c*beta) for beta in (0,1). Return 0 if c <= 1."""
    # YOUR CODE HERE
    pass

# (b) Compute L1/n for c values
n = 1000
c_vals = [0.5, 0.8, 1.0, 1.5, 2.0, 3.0]
L1_fracs = []

np.random.seed(42)
for c in c_vals:
    A = gnp_sample(n, c/n)
    L1 = bfs_largest_component(A)
    L1_fracs.append(L1 / n)

# (c) Theoretical values
beta_vals = [beta_theory(c) for c in c_vals]

print('c     L1/n_sim   beta_theory')
for c, l1, b in zip(c_vals, L1_fracs, beta_vals):
    print(f'{c:.1f}   {l1:.3f}      {b:.3f}')

In [ ]:
# Exercise 1 — SOLUTION

header('Exercise 1: Phase Transition Simulation', '★')

def gnp_sample(n, p, seed=None):
    rng = np.random.default_rng(seed)
    U = rng.random((n, n))
    A = (np.triu(U, k=1) < p).astype(float)
    return A + A.T

def bfs_largest_component(A):
    n = A.shape[0]
    visited = np.zeros(n, dtype=bool)
    max_size = 0
    for start in range(n):
        if visited[start]:
            continue
        queue = [start]
        visited[start] = True
        size = 0
        while queue:
            v = queue.pop()
            size += 1
            for u in np.where(A[v] > 0)[0]:
                if not visited[u]:
                    visited[u] = True
                    queue.append(u)
        max_size = max(max_size, size)
    return max_size

def beta_theory(c):
    if c <= 1:
        return 0.0
    return brentq(lambda b: b - 1 + np.exp(-c*b), 1e-10, 1.0 - 1e-10)

n = 1000
c_vals = [0.5, 0.8, 1.0, 1.5, 2.0, 3.0]
np.random.seed(42)
L1_fracs = [bfs_largest_component(gnp_sample(n, c/n)) / n for c in c_vals]
beta_vals = [beta_theory(c) for c in c_vals]

print('c     L1/n_sim   beta_theory')
for c, l1, b in zip(c_vals, L1_fracs, beta_vals):
    print(f'{c:.1f}   {l1:.3f}      {b:.3f}')

for c, l1, b in zip([1.5, 2.0, 3.0], L1_fracs[3:], beta_vals[3:]):
    check_close(l1, b, f'L1/n at c={c}', atol=0.05)

print('\nTakeaway: Giant component fraction matches theory beta(c) = 1 - exp(-c*beta(c)).')
print('  Phase transition at c=1: subcritical -> supercritical.')

---

## Exercise 2 ★ — Degree Distribution Comparison

**(a)** Generate $G(5000, 5/n)$ and compute the empirical degree distribution. Overlay the Poisson(5) PMF.

**(b)** Generate a BA graph with $n = 5000$, $m = 3$. Compute the empirical degree distribution on a log-log scale. Fit a power law: compute the MLE exponent $\hat{\gamma} = 1 + n \left[\sum_i \ln(k_i/(k_{\min} - 0.5))\right]^{-1}$ for $k_{\min} = 10$.

**(c)** Report: theoretical exponents for both models.

**(d)** What is the maximum degree in each model? Compare to the theoretical predictions.

In [ ]:
# Exercise 2 — YOUR CODE HERE

def barabasi_albert(n, m, seed=None):
    """BA preferential attachment. Returns adjacency matrix."""
    # YOUR CODE HERE
    pass

def mle_power_law_exponent(degrees, k_min):
    """Hill estimator for power law exponent."""
    # YOUR CODE HERE
    pass

# (a) ER degree distribution
n, c = 3000, 5.0
A_er = gnp_sample(n, c/n, seed=42)
deg_er = A_er.sum(axis=1).astype(int)
# YOUR CODE HERE: compare to Poisson(5)

# (b) BA degree distribution
n_ba, m_ba = 3000, 3
A_ba = barabasi_albert(n_ba, m_ba, seed=42)
deg_ba = A_ba.sum(axis=1).astype(int)
gamma_hat = mle_power_law_exponent(deg_ba, k_min=10)
print(f'BA MLE exponent: {gamma_hat:.3f} (theory: 3.0)')

In [ ]:
# Exercise 2 — SOLUTION

header('Exercise 2: Degree Distribution Comparison', '★')

from scipy.stats import poisson

def barabasi_albert(n, m, seed=None):
    rng = np.random.default_rng(seed)
    m0 = m + 1
    A = np.zeros((n, n))
    for i in range(m0):
        for j in range(i+1, m0):
            A[i,j] = A[j,i] = 1
    degrees = list(A[:m0].sum(axis=1)) + [0.0]*(n-m0)
    for new_node in range(m0, n):
        d_arr = np.array(degrees[:new_node])
        probs = d_arr / d_arr.sum()
        targets = rng.choice(new_node, size=m, replace=False, p=probs)
        for t in targets:
            A[new_node,t] = A[t,new_node] = 1
            degrees[new_node] += 1
            degrees[t] += 1
    return A

def mle_power_law_exponent(degrees, k_min):
    tail = degrees[degrees >= k_min]
    return 1 + len(tail) / np.sum(np.log(tail / (k_min - 0.5)))

# (a) ER
n, c = 3000, 5.0
A_er = gnp_sample(n, c/n, seed=42)
deg_er = A_er.sum(axis=1).astype(int)

check_close(deg_er.mean(), c, 'ER mean degree = c', atol=0.3)
check_close(deg_er.var(), c, 'ER degree variance ~ c (Poisson)', atol=0.5)

# (b) BA
n_ba, m_ba = 3000, 3
A_ba = barabasi_albert(n_ba, m_ba, seed=42)
deg_ba = A_ba.sum(axis=1).astype(int)

gamma_hat = mle_power_law_exponent(deg_ba, k_min=10)
check_close(gamma_hat, 3.0, 'BA power law exponent', atol=0.6)

# Max degrees
max_er = deg_er.max()
max_ba = deg_ba.max()
theory_max_ba = int(m_ba * np.sqrt(n_ba))
check_true(max_ba > max_er, f'BA max_deg ({max_ba}) > ER max_deg ({max_er})')
print(f'\nBA max degree: {max_ba} (theory ~m*sqrt(n) = {theory_max_ba})')

print('\nTakeaway: ER has Poisson (light tail, no hubs); BA has power-law (heavy tail, hubs).')

---

## Exercise 3 ★ — Watts-Strogatz Small-World

**(a)** Implement the Watts-Strogatz model: ring lattice + rewiring.

**(b)** Compute $C(\beta)$ and $L(\beta)$ for $\beta \in \{0, 0.001, 0.01, 0.05, 0.1, 0.5\}$ with $n=300$, $k=10$.

**(c)** Identify the small-world regime: which $\beta$ gives $C(\beta)/C(0) > 0.7$ AND $L(\beta)/L(0) < 0.5$?

**(d)** Verify the clustering formula: $C(\beta) \approx C(0)(1-\beta)^3$.

In [ ]:
# Exercise 3 — YOUR CODE HERE

def build_ring_lattice(n, k):
    """Ring lattice: each node connected to k nearest neighbors."""
    # YOUR CODE HERE
    pass

def rewire(A, k, beta, seed=None):
    """Rewire edges with probability beta."""
    # YOUR CODE HERE
    pass

def local_clustering(A):
    """Average local clustering coefficient."""
    # YOUR CODE HERE
    pass

def avg_path_length(A, n_samples=80):
    """Estimate average path length via BFS."""
    # YOUR CODE HERE
    pass

n_ws, k_ws = 300, 10
beta_vals = [0, 0.001, 0.01, 0.05, 0.1, 0.5]
# YOUR CODE HERE: compute C and L for each beta

In [ ]:
# Exercise 3 — SOLUTION

header('Exercise 3: Watts-Strogatz Small-World', '★')

def build_ring_lattice(n, k):
    A = np.zeros((n, n))
    for i in range(n):
        for j in range(1, k//2 + 1):
            u = (i + j) % n
            A[i, u] = A[u, i] = 1
    return A

def watts_strogatz(n, k, beta, seed=None):
    rng = np.random.default_rng(seed)
    A = build_ring_lattice(n, k)
    for i in range(n):
        for j in range(1, k//2 + 1):
            u = (i + j) % n
            if A[i, u] == 0:
                continue
            if rng.random() < beta:
                A[i, u] = A[u, i] = 0
                candidates = [v for v in range(n) if v != i and A[i, v] == 0]
                if candidates:
                    v_new = rng.choice(candidates)
                    A[i, v_new] = A[v_new, i] = 1
    return A

def local_clustering(A):
    n = A.shape[0]
    Cs = []
    for v in range(n):
        nbrs = np.where(A[v] > 0)[0]
        kv = len(nbrs)
        if kv < 2:
            Cs.append(0.0)
            continue
        tri = A[np.ix_(nbrs, nbrs)].sum() / 2
        Cs.append(tri / (kv*(kv-1)/2))
    return np.mean(Cs)

def avg_path_length(A, n_samples=80):
    n = A.shape[0]
    srcs = np.random.choice(n, min(n_samples, n), replace=False)
    total, cnt = 0, 0
    for s in srcs:
        dist = {s: 0}
        q = [s]
        for v in q:
            for u in np.where(A[v] > 0)[0]:
                if u not in dist:
                    dist[u] = dist[v] + 1
                    q.append(u)
        for d in dist.values():
            if d > 0:
                total += d
                cnt += 1
    return total / cnt if cnt > 0 else float('inf')

n_ws, k_ws = 300, 10
beta_vals = [0, 0.001, 0.01, 0.05, 0.1, 0.5]
C0 = 3*(k_ws-2)/(4*(k_ws-1))
L0 = n_ws / (2*k_ws)

print(f'Ring lattice: C0={C0:.4f}, L0={L0:.1f}')
print(f'{"beta":>8}  {"C(b)":>8}  {"C/C0":>6}  {"L(b)":>8}  {"L/L0":>6}')
np.random.seed(42)
small_world = []
for beta in beta_vals:
    A_ws = watts_strogatz(n_ws, k_ws, beta, seed=int(beta*10000+1))
    C = local_clustering(A_ws)
    L = avg_path_length(A_ws)
    print(f'{beta:8.3f}  {C:8.4f}  {C/C0:6.3f}  {L:8.2f}  {L/L0:6.3f}')
    if C/C0 > 0.7 and L/L0 < 0.5:
        small_world.append(beta)

print(f'\nSmall-world regime (C/C0>0.7, L/L0<0.5): beta={small_world}')

# (d) Verify clustering formula
beta_test = 0.05
A_test = watts_strogatz(n_ws, k_ws, beta_test, seed=42)
C_test = local_clustering(A_test)
C_theory_test = C0 * (1 - beta_test)**3
check_close(C_test, C_theory_test, f'C(beta={beta_test}) vs C0*(1-beta)^3', atol=0.05)
print('\nTakeaway: Small-world regime achieves HIGH clustering AND SHORT paths simultaneously.')

---

## Exercise 4 ★★ — Kesten-Stigum Threshold

The 2-block SBM with $p = a/n$, $q = b/n$ has a community detection threshold at $(a-b)^2 = 2(a+b)$.

**(a)** For fixed $a + b = 20$, sweep $a - b$ from 0 to 15. Compute the SNR $(a-b)^2/(2(a+b))$.

**(b)** For each $(a,b)$, generate an SBM with $n=400$ nodes and apply spectral clustering. Record accuracy.

**(c)** Plot accuracy vs SNR. Identify the phase transition.

**(d)** Is the phase transition sharp? How does it compare to the theoretical Kesten-Stigum threshold at SNR = 1?

In [ ]:
# Exercise 4 — YOUR CODE HERE

def sbm_2block(n, a, b, seed=None):
    """2-block SBM with equal communities, p=a/n, q=b/n."""
    # YOUR CODE HERE
    pass

def spectral_2block(A):
    """Spectral clustering: return estimated community labels."""
    # YOUR CODE HERE
    pass

def community_accuracy(est, true):
    """Accuracy up to global flip."""
    return max((est == true).mean(), (est == (1-true)).mean())

# YOUR CODE HERE: sweep diff = a - b while keeping a + b = 20

In [ ]:
# Exercise 4 — SOLUTION

header('Exercise 4: Kesten-Stigum Threshold', '★★')

def sbm_2block(n, a, b, seed=None):
    rng = np.random.default_rng(seed)
    labels = np.array([0]*(n//2) + [1]*(n//2))
    A = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            p = a/n if labels[i] == labels[j] else b/n
            if rng.random() < p:
                A[i,j] = A[j,i] = 1
    return A, labels

def spectral_2block(A):
    n = A.shape[0]
    d = A.sum(axis=1)
    D_isq = np.diag(np.where(d > 0, 1/np.sqrt(d), 0))
    L_sym = np.eye(n) - D_isq @ A @ D_isq
    eigvals, eigvecs = np.linalg.eigh(L_sym)
    return (eigvecs[:, 1] > 0).astype(int)

def community_accuracy(est, true):
    return max((est == true).mean(), (est == (1-true)).mean())

n_ks = 400
sum_ab = 20
diffs = np.linspace(0.5, 15, 18)
accs, snrs = [], []

np.random.seed(42)
for diff in diffs:
    a = (sum_ab + diff) / 2
    b = (sum_ab - diff) / 2
    if b <= 0:
        continue
    A_ks, lab_ks = sbm_2block(n_ks, a, b, seed=42)
    est_ks = spectral_2block(A_ks)
    accs.append(community_accuracy(est_ks, lab_ks))
    snrs.append(diff**2 / (2*sum_ab))

ks_snr = 1.0
ks_diff = np.sqrt(2*sum_ab)  # = sqrt(40) ~ 6.32
print(f'Kesten-Stigum threshold: diff = {ks_diff:.2f}, SNR = {ks_snr:.1f}')
print(f'\nSNR vs Accuracy:')
for s, a in zip(snrs[::3], accs[::3]):
    ks_mark = '<-- KS' if abs(s - 1.0) < 0.2 else ''
    print(f'  SNR={s:.2f}: acc={a:.3f} {ks_mark}')

# Check: above KS (SNR~2): acc > 0.7; below KS (SNR~0.3): acc ~ 0.5
above_idx = np.argmin(abs(np.array(snrs) - 2.0))
below_idx = np.argmin(abs(np.array(snrs) - 0.3))
check_true(accs[above_idx] > 0.70, f'Above KS (SNR~2): acc={accs[above_idx]:.3f} > 0.70')
check_true(accs[below_idx] < 0.65, f'Below KS (SNR~0.3): acc={accs[below_idx]:.3f} < 0.65')
print('\nTakeaway: KS threshold is sharp. No spectral algorithm beats random guessing below SNR=1.')

---

## Exercise 5 ★★ — Wigner Semicircle Law

**(a)** Generate Wigner GOE matrices $W_n = (M+M^\top)/(2\sqrt{n})$ for $n \in \{100, 500, 1000\}$. Plot the empirical spectral distribution and overlay the semicircle $\frac{2}{\pi}\sqrt{4-x^2}$.

**(b)** Verify: all eigenvalues of $W_{1000}$ lie in $[-2.1, 2.1]$ w.h.p.

**(c)** For $G(n,p)$ with $n=500$, $p=0.2$: center and scale $A$ to get $\hat{W} = (A - p\mathbf{1}\mathbf{1}^\top) / \sqrt{np(1-p)}$. Show the bulk eigenvalues follow the semicircle and identify the outlier.

In [ ]:
# Exercise 5 — YOUR CODE HERE

def wigner_goe(n, seed=None):
    """GOE Wigner matrix W = (M + M.T) / (2*sqrt(n))."""
    # YOUR CODE HERE
    pass

def semicircle(x):
    """Semicircle density on [-2, 2] with unit variance."""
    # YOUR CODE HERE
    pass

# (a) Test for n=100, 500, 1000
for n_w in [100, 500, 1000]:
    W = wigner_goe(n_w, seed=42)
    eigvals = np.linalg.eigvalsh(W)
    print(f'n={n_w}: max|lambda|={max(abs(eigvals)):.4f} (theory ~2.0)')

In [ ]:
# Exercise 5 — SOLUTION

header('Exercise 5: Wigner Semicircle Law', '★★')

def wigner_goe(n, seed=None):
    rng = np.random.default_rng(seed)
    M = rng.standard_normal((n, n))
    return (M + M.T) / (2 * np.sqrt(n))

def semicircle(x, R=2.0):
    mask = np.abs(x) <= R
    d = np.zeros_like(x, dtype=float)
    d[mask] = (2 / (np.pi * R**2)) * np.sqrt(R**2 - x[mask]**2)
    return d

# (a) & (b) Verify eigenvalue range
for n_w in [100, 500, 1000]:
    W = wigner_goe(n_w, seed=42)
    eigvals = np.linalg.eigvalsh(W)
    max_abs = max(abs(eigvals))
    ok = max_abs < 2.3
    print(f'n={n_w:5d}: max|lambda|={max_abs:.4f} (theory ~2.0)  {"PASS" if ok else "FAIL"}')

check_true(max_abs < 2.3, f'Max eigenvalue < 2.3 for n=1000')

# (c) G(n,p) bulk vs outlier
n_gp, p_gp = 500, 0.2
A_gp = gnp_sample(n_gp, p_gp, seed=42)
# Center (remove mean matrix)
sigma = np.sqrt(n_gp * p_gp * (1-p_gp))
A_centered = A_gp - p_gp * (np.ones((n_gp, n_gp)) - np.eye(n_gp))
W_gp = A_centered / sigma

eigvals_gp = np.linalg.eigvalsh(A_gp)
outlier = eigvals_gp[-1]
theory_outlier = n_gp * p_gp
bulk_radius = 2 * sigma

print(f'\nG(n,p) outlier: {outlier:.2f} (theory np={theory_outlier:.0f})')
print(f'Bulk radius 2*sigma: {bulk_radius:.2f}')
check_close(outlier, theory_outlier, 'Outlier eigenvalue ~ np', atol=theory_outlier*0.1)
print('\nTakeaway: Wigner bulk follows semicircle; G(n,p) has one outlier eigenvalue at ~np.')

---

## Exercise 6 ★★ — Graphon Estimation

**(a)** Generate a 3-block SBM graph ($n = 300$) with block matrix $B = \begin{pmatrix} 0.8 & 0.1 & 0.1 \\ 0.1 & 0.7 & 0.2 \\ 0.1 & 0.2 & 0.6 \end{pmatrix}$.

**(b)** Sort nodes by true community label. Display the sorted adjacency matrix as a heatmap. How well does it reveal block structure?

**(c)** Apply spectral clustering (3 communities) to estimate labels. Display the estimated sorted adjacency matrix. How does block structure compare to oracle sorting?

**(d)** Compute the $L^2$ distance between the estimated graphon and the true step-function graphon:
$$\|\hat{W} - W^*\|_{L^2}^2 = \int_0^1 \int_0^1 (\hat{W}(x,y) - W^*(x,y))^2 \, dx \, dy$$

In [ ]:
# Exercise 6 — YOUR CODE HERE

B_true = np.array([[0.8, 0.1, 0.1],
                   [0.1, 0.7, 0.2],
                   [0.1, 0.2, 0.6]])

def sbm_3block(n, B, seed=None):
    """3-block SBM with equal communities."""
    # YOUR CODE HERE
    pass

def spectral_clustering_k(A, k):
    """k-community spectral clustering via normalized Laplacian top k eigenvectors + k-means."""
    # YOUR CODE HERE
    pass

def estimate_graphon(A, labels, k):
    """Estimate block-graphon from sorted adjacency matrix."""
    # YOUR CODE HERE
    pass

# YOUR CODE HERE: generate, sort, display, estimate

In [ ]:
# Exercise 6 — SOLUTION

header('Exercise 6: Graphon Estimation', '★★')

B_true = np.array([[0.8, 0.1, 0.1],
                   [0.1, 0.7, 0.2],
                   [0.1, 0.2, 0.6]])

def sbm_3block(n, B, seed=None):
    rng = np.random.default_rng(seed)
    k = B.shape[0]
    labels = np.repeat(np.arange(k), n // k)
    n_act = len(labels)
    A = np.zeros((n_act, n_act))
    for i in range(n_act):
        for j in range(i+1, n_act):
            p = B[labels[i], labels[j]]
            if rng.random() < p:
                A[i,j] = A[j,i] = 1
    return A, labels

def spectral_clustering_k(A, k):
    from sklearn.cluster import KMeans
    n = A.shape[0]
    d = A.sum(axis=1)
    D_isq = np.diag(np.where(d > 0, 1/np.sqrt(d), 0))
    L_sym = np.eye(n) - D_isq @ A @ D_isq
    eigvals, eigvecs = np.linalg.eigh(L_sym)
    U = eigvecs[:, :k]  # k smallest eigenvectors
    norms = np.linalg.norm(U, axis=1, keepdims=True)
    U_norm = U / (norms + 1e-10)
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    return km.fit_predict(U_norm)

def estimate_graphon(A, labels, k):
    n = len(labels)
    block_size = n // k
    B_est = np.zeros((k, k))
    for r in range(k):
        for s in range(k):
            r_idx = np.where(labels == r)[0]
            s_idx = np.where(labels == s)[0]
            if r == s:
                sub = A[np.ix_(r_idx, r_idx)]
                n_r = len(r_idx)
                denom = n_r * (n_r - 1)
            else:
                sub = A[np.ix_(r_idx, s_idx)]
                denom = len(r_idx) * len(s_idx)
            B_est[r, s] = sub.sum() / max(denom, 1)
    return B_est

n_gon = 300
np.random.seed(42)
A_gon, lab_true_gon = sbm_3block(n_gon, B_true, seed=42)

# (c) Estimated labels
try:
    lab_est_gon = spectral_clustering_k(A_gon, 3)
except ImportError:
    # Fallback without sklearn: use top eigenvectors + threshold
    d_g = A_gon.sum(axis=1)
    D_isq_g = np.diag(np.where(d_g > 0, 1/np.sqrt(d_g), 0))
    L_g = np.eye(len(A_gon)) - D_isq_g @ A_gon @ D_isq_g
    eigvals_g, eigvecs_g = np.linalg.eigh(L_g)
    # Simple: use signs of 2nd and 3rd eigenvectors
    u2, u3 = eigvecs_g[:, 1], eigvecs_g[:, 2]
    lab_est_gon = (u2 > 0).astype(int) + (u3 > 0).astype(int)

# (d) Estimate graphons
B_est_oracle = estimate_graphon(A_gon, lab_true_gon, 3)
B_est_spectral = estimate_graphon(A_gon, lab_est_gon, 3)

print('Oracle estimated B:')
print(B_est_oracle.round(3))
print('\nTrue B:')
print(B_true.round(3))

# L2 error (step graphon: integral = sum over blocks weighted by block area)
k = 3
block_area = (1/k)**2
l2_oracle = np.sqrt(block_area * np.sum((B_est_oracle - B_true)**2))
print(f'\nOracle graphon L2 error: {l2_oracle:.4f}')

check_true(l2_oracle < 0.15, f'Oracle graphon L2 error < 0.15 (actual: {l2_oracle:.4f})')
print('\nTakeaway: Sorted adjacency matrix converges to step-function graphon as n→∞.')

---

## Exercise 7 ★★★ — Critical Window of the Phase Transition

The critical window of the ER phase transition scales as $n^{1/3}$ in $p$ and $n^{2/3}$ in component size.

**(a)** For $p = (1 + \lambda n^{-1/3})/n$ with $\lambda \in [-3, 3]$ and $n \in \{300, 1000, 3000\}$, compute $L_1/n^{2/3}$ for 30 trials each.

**(b)** Plot $\mathbb{E}[L_1/n^{2/3}]$ vs $\lambda$ for all three $n$ values. Do the curves converge as $n$ grows?

**(c)** For $\lambda = 2$ (supercritical), verify that $\mathbb{E}[L_1] = O(n)$ while $\text{Std}[L_1] = O(n^{2/3})$.

**(d)** The second moment of $L_1/n^{2/3}$ near $\lambda = 0$ should be approximately $\mathbb{E}[(L_1/n^{2/3})^2] \approx 1$. Verify this numerically.

In [ ]:
# Exercise 7 — YOUR CODE HERE

# Critical window analysis
# p = (1 + lambda * n^{-1/3}) / n

lambdas = np.linspace(-3, 3, 13)
ns = [300, 1000, 3000]
n_trials = 20

# YOUR CODE HERE: simulate and collect L1/n^{2/3} for each (n, lambda)

In [ ]:
# Exercise 7 — SOLUTION

header('Exercise 7: Critical Window of Phase Transition', '★★★')

lambdas = np.linspace(-3, 3, 13)
ns = [300, 1000, 3000]
n_trials = 20

results = {}  # (n, lambda) -> list of L1/n^{2/3}

np.random.seed(42)
for n_crit in ns:
    for lam in lambdas:
        p_crit = (1 + lam * n_crit**(-1/3)) / n_crit
        p_crit = max(0, min(1, p_crit))
        L1_scaled = []
        for _ in range(n_trials):
            A_crit = gnp_sample(n_crit, p_crit)
            L1 = bfs_largest_component(A_crit)
            L1_scaled.append(L1 / (n_crit**(2/3)))
        results[(n_crit, lam)] = L1_scaled

# Print mean L1/n^{2/3} for n=1000
n_show = 1000
print(f'n={n_show}: E[L1/n^{{2/3}}] vs lambda')
for lam in lambdas[::3]:
    mean_L1 = np.mean(results[(n_show, lam)])
    print(f'  lambda={lam:+.1f}: {mean_L1:.3f}')

# (c) Variance check at lambda=2
lam2 = lambdas[np.argmin(abs(lambdas - 2.0))]
for n_v in ns:
    raw = [v * n_v**(2/3) for v in results[(n_v, lam2)]]
    mean_raw = np.mean(raw)
    std_raw = np.std(raw)
    scale_L1 = mean_raw / n_v
    scale_std = std_raw / n_v**(2/3)
    print(f'n={n_v}, lambda={lam2:.1f}: E[L1]/n={scale_L1:.4f}, Std[L1]/n^{{2/3}}={scale_std:.4f}')

# (d) Second moment near lambda=0
lam0 = lambdas[np.argmin(abs(lambdas - 0.0))]
for n_s in ns:
    second_moment = np.mean(np.array(results[(n_s, lam0)])**2)
    print(f'n={n_s}, lambda~0: E[(L1/n^{{2/3}})^2] = {second_moment:.3f} (expect ~1)')

print('\nTakeaway: Critical window has n^{{1/3}} width in p and n^{{2/3}} scaling in L1.')
print('  E[L1/n^{{2/3}}] curves converge as n grows — universal critical exponent.')

---

## Exercise 8 ★★★ — Mean-Field Preferential Attachment with Fitness

Extend the BA model with per-node fitness $\eta_v \sim \text{Uniform}[0,1]$: $\Pi(v) \propto \eta_v \deg(v)$.

**(a)** Implement the fitness-based BA model.

**(b)** Show that the degree of node $i$ (with fitness $\eta_i$ added at time $t_i$) satisfies $k_i(t) \approx m(t/t_i)^{\eta_i}$ in the mean-field approximation.

**(c)** Fit the degree distribution of the fitness model. Does it still follow a power law? What is the exponent?

**(d)** Compare robustness: fitness-BA vs standard-BA vs ER. Which is most robust to targeted (high-degree) attack? Which is most robust to random failure?

In [ ]:
# Exercise 8 — YOUR CODE HERE

def fitness_ba(n, m, seed=None):
    """Fitness-based BA: Pi(v) proportional to fitness[v] * degree[v]."""
    # YOUR CODE HERE
    pass

n_fit = 2000
m_fit = 2
# YOUR CODE HERE: generate, fit power law, compare robustness

In [ ]:
# Exercise 8 — SOLUTION

header('Exercise 8: Fitness-Based Preferential Attachment', '★★★')

def fitness_ba(n, m, seed=None):
    """Fitness-based BA."""
    rng = np.random.default_rng(seed)
    m0 = m + 1
    # Fitness for all nodes
    fitness = rng.uniform(0.5, 1.5, n)  # Uniform[0.5, 1.5]
    A = np.zeros((n, n))
    for i in range(m0):
        for j in range(i+1, m0):
            A[i,j] = A[j,i] = 1
    degrees = list(A[:m0].sum(axis=1)) + [0.0]*(n-m0)

    for new_node in range(m0, n):
        d_arr = np.array(degrees[:new_node])
        f_arr = fitness[:new_node]
        weights = f_arr * (d_arr + 1e-10)  # fitness * degree
        probs = weights / weights.sum()
        targets = rng.choice(new_node, size=m, replace=False, p=probs)
        for t in targets:
            A[new_node, t] = A[t, new_node] = 1
            degrees[new_node] += 1
            degrees[t] += 1
    return A, fitness

n_fit, m_fit = 2000, 2
np.random.seed(42)
A_fit, fitness_fit = fitness_ba(n_fit, m_fit, seed=42)
deg_fit = A_fit.sum(axis=1).astype(int)

# Standard BA for comparison
A_ba_cmp = barabasi_albert(n_fit, m_fit, seed=42)
deg_ba_cmp = A_ba_cmp.sum(axis=1).astype(int)

# Power law fit
def mle_power_law(degrees, k_min):
    tail = degrees[degrees >= k_min]
    if len(tail) == 0:
        return float('nan')
    return 1 + len(tail) / np.sum(np.log(tail / (k_min - 0.5)))

gamma_fit = mle_power_law(deg_fit, k_min=8)
gamma_ba = mle_power_law(deg_ba_cmp, k_min=8)

print(f'Standard BA: gamma = {gamma_ba:.3f} (theory: 3.0)')
print(f'Fitness BA:  gamma = {gamma_fit:.3f}')
print(f'Max degree - Standard: {deg_ba_cmp.max()}, Fitness: {deg_fit.max()}')

# (b) Mean-field check: k_i(t) ~ m * (t/t_i)^{eta_i}
# For node i=10 (added early), check its degree
node_i = 10
eta_i = fitness_fit[node_i]
t_i = node_i
t_final = n_fit
theory_k = m_fit * (t_final / t_i)**eta_i
actual_k = deg_fit[node_i]
print(f'\nNode {node_i}: fitness={eta_i:.3f}, actual_k={actual_k}, '
      f'mean_field_k={theory_k:.1f}')

# (d) Robustness comparison
fracs_rob = [0.0, 0.05, 0.1, 0.2, 0.3]
n_rob = 300
A_ba_rob = barabasi_albert(n_rob, 2, seed=42)
A_er_rob = gnp_sample(n_rob, 4/n_rob, seed=42)

def targeted_attack_gc(A, f):
    n = A.shape[0]
    deg = A.sum(axis=1)
    remove = set(np.argsort(-deg)[:int(f*n)])
    idx = [i for i in range(n) if i not in remove]
    if not idx:
        return 0
    A_sub = A[np.ix_(idx, idx)]
    L1, _ = bfs_largest_component(A_sub), 0
    return bfs_largest_component(A_sub) / n

print('\nTargeted attack: giant component fraction')
print(f'{"f":>6}  {"BA":>6}  {"ER":>6}')
for f in fracs_rob:
    gc_ba = targeted_attack_gc(A_ba_rob, f)
    gc_er = targeted_attack_gc(A_er_rob, f)
    print(f'{f:6.2f}  {gc_ba:6.3f}  {gc_er:6.3f}')

print('\nTakeaway: Fitness-BA has similar power-law exponent to standard BA but ')
print('  high-fitness nodes become hubs regardless of arrival time.')
print('  BA is robust to random failure but fragile to targeted hub removal.')